# GPU transformer benchmark

This Colab workflow compares a compact DistilBERT model with the classical baseline on the **same five connected-group outer folds**. Enable a GPU runtime before running it. The command writes fold-level JSON evidence; it does not overwrite the published classical benchmark.

Expected cost on a free GPU is hardware- and quota-dependent. Start with two folds for a smoke test, then run all five for publishable evidence. Free Colab sessions can disconnect, so download the JSON result immediately.

In [ ]:
!git clone https://github.com/kristishqau/SentimentAnalysis_NLP.git
%cd SentimentAnalysis_NLP
%pip install -q -e ".[transformer]" "scikit-learn==1.9.1"

In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## Smoke test

Two grouped folds verify the environment. This is not the final comparison.

In [ ]:
!sentiment-transformer-benchmark \
  --folds 2 --epochs 1 --batch-size 32 \
  --output artifacts/transformer_smoke.json

## Publishable five-fold comparison

Run these only after the smoke test succeeds. Both models use identical canonical outer folds, grouped inner validation, macro-F1 checkpoint selection, and balanced loss. If memory is limited, lower `--batch-size` to 16.

In [ ]:
!sentiment-transformer-benchmark \
  --folds 5 --epochs 3 --batch-size 32 \
  --validation-folds 5 --early-stopping-patience 1 \
  --class-weighting balanced \
  --reference reports/benchmark_v2.json \
  --output reports/transformer_distilbert_benchmark.json

In [ ]:
!sentiment-transformer-benchmark \
  --model cardiffnlp/twitter-roberta-base \
  --folds 5 --epochs 3 --batch-size 32 \
  --validation-folds 5 --early-stopping-patience 1 \
  --class-weighting balanced \
  --reference reports/benchmark_v2.json \
  --output reports/transformer_benchmark.json

In [ ]:
import json
from pathlib import Path

distilbert = json.loads(Path("reports/transformer_distilbert_benchmark.json").read_text())
twitter_roberta = json.loads(Path("reports/transformer_benchmark.json").read_text())
classical = json.loads(Path("reports/benchmark_v2.json").read_text())
print("DistilBERT macro-F1:", distilbert["summary"]["macro_f1"])
print("Twitter-RoBERTa macro-F1:", twitter_roberta["summary"]["macro_f1"])
print("Classical macro-F1:", classical["candidates"]["entity_word_char"]["summary"]["macro_f1"])

In [ ]:
from google.colab import files

files.download("reports/transformer_distilbert_benchmark.json")
files.download("reports/transformer_benchmark.json")

## Nested optimization study

This preregistered follow-up screens DistilBERT, Twitter-RoBERTa, DistilRoBERTa, and DeBERTa-v3-small inside each outer training partition. It then searches learning rates `1e-5`, `2e-5`, `3e-5`, lengths `96`, `128`, `192`, and weight decay `0.0`, `0.01` only for that fold's validation-selected encoder. Seeds 42, 43, and 44 measure stability; outer-test scores never select a seed.

Each fold is saved and downloaded immediately. If Colab disconnects, upload completed fold files, remove their numbers from `FOLDS_TO_RUN`, and continue. A full run is substantially more expensive than the fixed benchmarks above.

In [ ]:
import subprocess

from google.colab import files

FOLDS_TO_RUN = [1, 2, 3, 4, 5]
for fold in FOLDS_TO_RUN:
    output = f"artifacts/transformer_study_fold_{fold}.json"
    subprocess.run(
        [
            "sentiment-transformer-study",
            "--outer-fold",
            str(fold),
            "--epochs",
            "4",
            "--batch-size",
            "32",
            "--reference",
            "reports/benchmark_v2.json",
            "--output",
            output,
        ],
        check=True,
    )
    files.download(output)

## Merge and download final study evidence

All five fold files must be present. Upload any files retained from an earlier Colab session before running this cell. The merge command rejects missing, duplicate, dataset-mismatched, or protocol-mismatched folds.

In [ ]:
study_folds = [f"artifacts/transformer_study_fold_{fold}.json" for fold in range(1, 6)]
subprocess.run(
    [
        "sentiment-transformer-study-merge",
        *study_folds,
        "--output",
        "reports/transformer_optimization_study.json",
    ],
    check=True,
)
files.download("reports/transformer_optimization_study.json")